In [1]:
#weed out single-brain-celled emotionally incoherent stinking childish annoying nincompoops
from dataclasses import dataclass, field
from typing import Callable, Dict, List, Tuple, Any
import re

# -----------------------------
# Data model
# -----------------------------

@dataclass
class Prospect:
    """One person's interaction snapshot."""
    id: str
    name: str
    phrases: List[str] = field(default_factory=list)      # direct quotes / key phrases
    behaviors: List[str] = field(default_factory=list)    # normalized tokens: ["ghosting", "stalking", ...]
    notes: str = ""                                       # free text
    attraction: str = ""                                  # optional: "attracted"/"not attracted"
    nonneg_mismatch: bool = False                          # did non-neg mismatch happen?
    age_diff_years: int | None = None                      # optional


@dataclass
class RuleResult:
    rule_id: str
    points: int
    tags: List[str]
    reason: str


@dataclass
class EngineOutput:
    total_points: int
    label: str                  # "N" or "NN"
    risk_level: str             # "low" / "medium" / "high"
    tags: List[str]
    fired_rules: List[RuleResult]


# -----------------------------
# Rule Engine
# -----------------------------

@dataclass
class Rule:
    rule_id: str
    condition: Callable[[Prospect], bool]
    points: int
    tags: List[str]
    reason: str


class RuleEngine:
    def __init__(self, rules: List[Rule], threshold_nn: int = 6):
        """
        threshold_nn: points >= threshold_nn => NN
        """
        self.rules = rules
        self.threshold_nn = threshold_nn

    def evaluate(self, p: Prospect) -> EngineOutput:
        fired: List[RuleResult] = []
        total = 0
        tagset = set()

        for r in self.rules:
            if r.condition(p):
                total += r.points
                tagset.update(r.tags)
                fired.append(RuleResult(r.rule_id, r.points, r.tags, r.reason))

        label = "NN" if total >= self.threshold_nn else "N"

        # risk_level can be your own mapping
        if total >= 12:
            risk = "high"
        elif total >= 6:
            risk = "medium"
        else:
            risk = "low"

        return EngineOutput(
            total_points=total,
            label=label,
            risk_level=risk,
            tags=sorted(tagset),
            fired_rules=fired
        )


# -----------------------------
# Helpers (phrase/behavior matchers)
# -----------------------------

def any_phrase_matches(p: Prospect, patterns: List[str]) -> bool:
    text = " | ".join(p.phrases + [p.notes])
    text_lower = text.lower()
    return any(re.search(ptn, text_lower) for ptn in patterns)

def has_behavior(p: Prospect, behavior: str) -> bool:
    return behavior.lower() in {b.lower() for b in p.behaviors}


# -----------------------------
# Your rules (customize freely)
# -----------------------------

RULES: List[Rule] = [

    # Hard safety / ethics
    Rule(
        rule_id="SAFETY_STALKING",
        condition=lambda p: has_behavior(p, "stalking") or any_phrase_matches(p, [r"\bstalk", r"view(ed)? my linkedin.*many times"]),
        points=10,
        tags=["safety", "stalking", "boundary-violation"],
        reason="Stalking / repeated unwanted tracking is a hard safety red flag."
    ),
    Rule(
        rule_id="SAFETY_CATFISHING",
        condition=lambda p: has_behavior(p, "catfishing") or any_phrase_matches(p, [r"\bcatfish", r"changed (his )?age", r"identity (mis)?representation"]),
        points=10,
        tags=["safety", "deception", "catfishing"],
        reason="Identity/age manipulation suggests deception and risk."
    ),
    Rule(
        rule_id="SEXUAL_OBJECTIFICATION",
        condition=lambda p: has_behavior(p, "asked_for_nudes") or any_phrase_matches(p, [r"\bnudes?\b", r"naked pictures", r"sex is very important.*constantly"]),
        points=9,
        tags=["safety", "sexual-pressure", "objectification"],
        reason="Sexual coercion/objectification early is a major red flag."
    ),

    # Reliability / coherence
    Rule(
        rule_id="GHOSTING",
        condition=lambda p: has_behavior(p, "ghosting") or any_phrase_matches(p, [r"\bghost", r"stopped replying", r"disappeared for"]),
        points=6,
        tags=["reliability", "avoidant", "incoherent"],
        reason="Ghosting indicates poor communication and low reliability."
    ),
    Rule(
        rule_id="DODGE_NONNEG",
        condition=lambda p: has_behavior(p, "dodged_nonneg") or any_phrase_matches(p, [r"avoid(ed)?", r"deflect(ed)?", r"went silent", r"changed the topic"]),
        points=5,
        tags=["honesty", "avoidance", "nonneg-evasion"],
        reason="Dodging direct questions around non-negotiables is a common manipulation/avoidance pattern."
    ),

    # Character / worldview flags (tunable)
    Rule(
        rule_id="MISOGYNY_GENERALIZATION",
        condition=lambda p: has_behavior(p, "misogyny") or any_phrase_matches(p, [r"all women are bad", r"women are evil", r"men love completely"]),
        points=9,
        tags=["misogyny", "projection", "unhealed"],
        reason="Global hatred/projection toward women is unsafe and incompatible."
    ),
    Rule(
        rule_id="MORAL_HYPOCRISY",
        condition=lambda p: has_behavior(p, "hypocrisy") or any_phrase_matches(p, [r"values.*(but|while)", r"religio(us|n).*judg", r"pravachan", r"preach"]),
        points=6,
        tags=["hypocrisy", "performance", "integrity-risk"],
        reason="Preaching/virtue-signaling while living opposite values often maps to deception or low integrity."
    ),
    Rule(
        rule_id="CONTROL_VIA_HELP",
        condition=lambda p: has_behavior(p, "help_as_hook") or any_phrase_matches(p, [r"i am helping because you are a girl", r"you need help", r"reach out to my friend"]),
        points=4,
        tags=["control", "hook", "power-dynamic"],
        reason="Help framed as gender/power can become leverage. (Not always bad; treat as a watch-item.)"
    ),

    # Your specific "tell" phrases from logs (examples)
    Rule(
        rule_id="QUEUE_WOMEN_PHRASE",
        condition=lambda p: any_phrase_matches(p, [r"waiting for (women|girls) in a queue"]),
        points=8,
        tags=["entitlement", "objectification"],
        reason="Entitlement language about women indicates poor character."
    ),
    Rule(
        rule_id="ASKS_YOU_TO_REJECT",
        condition=lambda p: any_phrase_matches(p, [r"can you reject me\??"]),
        points=6,
        tags=["avoidance", "people-pleasing", "cowardice"],
        reason="Asking you to reject so he avoids accountability is a maturity red flag."
    ),

    # Soft positives (subtract points) — optional
    Rule(
        rule_id="POSITIVE_REFLECTIVE_EQ",
        condition=lambda p: has_behavior(p, "reflective") or any_phrase_matches(p, [r"growth mentality", r"self reflective", r"let'?s take time and decide"]),
        points=-3,
        tags=["green-flag", "eq"],
        reason="Reflective language + slowing down can indicate emotional regulation."
    ),
    Rule(
        rule_id="POSITIVE_DIRECTNESS",
        condition=lambda p: has_behavior(p, "direct") or any_phrase_matches(p, [r"this is not a random conversation", r"i propose we take time"]),
        points=-2,
        tags=["green-flag", "directness"],
        reason="Directness and clarity reduce risk."
    ),
]


# -----------------------------
# Example usage
# -----------------------------
if __name__ == "__main__":
    engine = RuleEngine(RULES, threshold_nn=6)

    p6 = Prospect(
        id="6",
        name="Ghosting Performing-Empathy Catfisher",
        phrases=[
            "let’s see where this goes if it has to happen it will",
            "I’m not telling anything to my kids about what I did",
            "alcohol? Of course Bangalore",
        ],
        behaviors=["ghosting", "catfishing", "help_as_hook"],
        notes="Changed age, targeted 18–33, repeated ghosting after PCOS."
    )

    out = engine.evaluate(p6)
    print("Total:", out.total_points, "Label:", out.label, "Risk:", out.risk_level)
    print("Tags:", out.tags)
    print("\nFired rules:")
    for rr in out.fired_rules:
        print("-", rr.rule_id, rr.points, rr.tags, "=>", rr.reason)


Total: 20 Label: NN Risk: high
Tags: ['avoidant', 'catfishing', 'control', 'deception', 'hook', 'incoherent', 'power-dynamic', 'reliability', 'safety']

Fired rules:
- SAFETY_CATFISHING 10 ['safety', 'deception', 'catfishing'] => Identity/age manipulation suggests deception and risk.
- GHOSTING 6 ['reliability', 'avoidant', 'incoherent'] => Ghosting indicates poor communication and low reliability.
- CONTROL_VIA_HELP 4 ['control', 'hook', 'power-dynamic'] => Help framed as gender/power can become leverage. (Not always bad; treat as a watch-item.)
